In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

from src.services.gec.config import CHECKPOINT_PATH, LABEL2ID_PATH
from src.services.gec.training import GECTrainingDataset, build_trainer, create_model
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = Path("./gec_models/edit_tagger_v1")
EPOCHS = 3
BATCH_SIZE = 8
LR = 3e-5
MAX_LEN = 256

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(LABEL2ID_PATH) as f:
    label2id = json.load(f)

print(f"Labels: {len(label2id)}, Examples: {sum(1 for _ in open(CHECKPOINT_PATH))}")

In [ ]:
dataset = GECTrainingDataset(CHECKPOINT_PATH, tokenizer, label2id, max_length=MAX_LEN)
sample = dataset[0]
assert len(sample["input_ids"]) == len(sample["labels"])
print(f"Dataset: {len(dataset)} examples, sample len: {len(sample[\"input_ids\"])}")

In [ ]:
model = create_model(MODEL_NAME, label2id)

In [ ]:
trainer = build_trainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    label2id_path=LABEL2ID_PATH,
)

In [ ]:
trainer.train()

In [ ]:
save_path = OUTPUT_DIR / "best"
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f"Saved to {save_path}")